In [ ]:
# %% [markdown]
# Notebook ETL – leitura em batch, validações e write-back


In [ ]:
# %% [code]
import os
import sys
import logging
from dotenv import load_dotenv
from rich.console import Console
from rich.logging import RichHandler

# ── Ajuste do PYTHONPATH para permitir importar de logs/ ──────────────────
project_root = os.getcwd()  # supondo que o notebook esteja em /home/debrito/Documentos/etl_debrito
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from logs.logging_setup import get_logger  # importa o configurador unificado

# ── Carrega variáveis de ambiente de .env (opcional) ───────────────────────
load_dotenv()

# ── Configuração de logs no notebook ──────────────────────────────────────
console = Console(width=120)
root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)

# ── Atenção: NÃO remover handlers existentes (por exemplo, o FileHandler) ──
#for h in root_logger.handlers[:]:
#    root_logger.removeHandler(h)

# ── Cria apenas o RichHandler para console, mantendo o FileHandler intacto ─
rich_handler = RichHandler(
    console=console,
    rich_tracebacks=True,
    show_time=True,
    show_level=True,
    show_path=False,
    markup=True,
)
rich_handler.setLevel(logging.DEBUG)
rich_handler.setFormatter(
    logging.Formatter("%(asctime)s %(levelname)s %(name)s › %(message)s", datefmt="%H:%M:%S")
)
root_logger.addHandler(rich_handler)

# ── Logger específico para este notebook ──────────────────────────────────
log = get_logger(__name__)
log.debug("Logger configurado para o notebook (RichHandler + FileHandler ativos)")


In [ ]:
#2 %% [code]
import math
import numpy as np
from typing import Any

def _to_json_safe(x: Any) -> Any:
    """
    Internal helper to convert arbitrary objects into JSON-safe primitives.
    """
    if x is None:
        return None
    if isinstance(x, (int, str, bool)):
        return x
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if (math.isnan(x) or math.isinf(x)) else float(x)
    if isinstance(x, (list, tuple, set)):
        return [_to_json_safe(item) for item in x]
    if isinstance(x, dict):
        return {k: _to_json_safe(v) for k, v in x.items()}
    # Fallback: stringify anything else
    return str(x)

def json_safe(obj: Any) -> Any:
    """
    Recursively converts `obj` into structures 100% serializable to JSON.
    """
    return _to_json_safe(obj)


In [ ]:
#3 %% [code]
# Flags de gravação (ajuste conforme necessidade)
WRITE_BACK_ORIGIN = True   # grava na aba-origem (meta*, tiktok*, …)
WRITE_BACK_DEST   = True   # grava nas abas-modelo (modelo*)
DRY_RUN_DEST      = False  # True = simula write-back destino

# Credenciais e identificador da planilha
CREDS_PATH     = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
SPREADSHEET_ID = os.getenv(
    "GOOGLE_SHEET_ID",
    "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
)

# Abas de origem a processar, agrupadas por plataforma
SHEET_NAMES = [
    # Meta (Facebook / Instagram)
    "metaGeral",
    "metaIdade",
    "metaGenero",
    "metaRegiao",
    "metaAlcance",
    # TikTok
    "tiktokGeral",
    "tiktokIdade",
    "tiktokGenero",
    "tiktokRegiao",
    "tiktokAlcance",
    # Pinterest
    "pinterestGeral",
    "pinterestGenero",
    "pinterestIdade",
    "pinterestRegiao",
    "pinterestAlcance",
    # LinkedIn
    "linkedinGeral",
    "linkedinRegiao",
    "linkedinAlcance",
    # Google Analytics
    "GAGeral",
]


In [ ]:
#4
# %% [code]
import importlib

# módulos principais e helpers recarregados
import extract.sheets_fetcher               as sf_mod
import treat.treat_pipeline                 as tp_mod
import treat.platforms                      as platforms_mod
import treat.platforms.linkedin             as linkedin_mod
import treat.platforms.tiktok               as tiktok_mod
import treat.platforms.pinterest            as pinterest_mod
import treat.platforms.meta                 as meta_mod
import treat.platforms.ga                   as ga_mod
import load.origin_writer                   as ow_mod
import load.dest_writer                     as dw_mod
import treat.utils.renomeacoes              as rn_mod
import treat.utils.preview_links            as prev_mod
import treat.utils.atribuicoes_via_lookup   as atrib_mod
import treat.utils.substitute_origin_values as sub_mod
import treat.utils.preprocess_utils         as pre_mod
import treat.utils.geo_normalize            as geo_mod

# hot-reload de todos os módulos alterados durante o desenvolvimento
for m in (
    sf_mod,
    tp_mod,
    platforms_mod,
    linkedin_mod,
    tiktok_mod,
    pinterest_mod,
    meta_mod,
    ga_mod,
    ow_mod,
    dw_mod,
    rn_mod,
    prev_mod,
    atrib_mod,
    sub_mod,
    pre_mod,
    geo_mod,
):
    importlib.reload(m)


In [ ]:
# %% [code]
# Cell 5: definição do helper run_etl_for_sheet
import pandas as pd
import gc
import json
from pprint import pp
from typing import Dict

from logs.logging_setup import get_logger
log = get_logger(__name__)

from extract.sheets_fetcher import SheetsFetcher
from treat.treat_pipeline import TreatPipeline
from treat.utils.renomeacoes import renomeacao_geral, renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total, gerar_id
from load.origin_writer import write_back_origin
from load.dest_writer import write_back_for_sheet

# Instância única do fetcher (retry/backoff/cache interno)
fetcher = SheetsFetcher(
    spreadsheet_id=SPREADSHEET_ID,
    creds_path=CREDS_PATH,
)

def run_etl_for_sheet(
    *,
    sheet: str,
    wb_origin_flag: bool,
    wb_dest_flag: bool,
    dry_run_dest: bool,
    preloaded_raw: pd.DataFrame,
) -> Dict[str, pd.DataFrame | dict]:
    """
    Executa o fluxo completo para uma aba e devolve:
      { "dest": DataFrame destino (ou vazio), "taxo": relatório de taxonomia }
    """
    # 1) Dados brutos já carregados
    df_raw = preloaded_raw

    # 2) Tratamento via pipeline
    pipeline = TreatPipeline(
        creds_path=CREDS_PATH,
        spreadsheet_id=SPREADSHEET_ID,
        sheet_name=sheet,
        mapping_renomeacao=renomeacao_geral,
        write_back=wb_origin_flag,
    )
    df_ok = pipeline.run(df_raw)

    # 3) Relatório de taxonomia
    taxo_report = getattr(pipeline, "_last_taxo_report", {})
    pp(json.dumps(taxo_report, default=str), width=120)

    # 4) Write-back na aba de origem (apenas quando não for Pinterest demográfico)
    is_pinterest_dim = sheet.lower() in {
        "pinterestgenero", "pinterestidade", "pinterestregiao"
    }

    if not is_pinterest_dim:
        # grava correções de pré-processamento in-place
        _ = write_back_origin(
            df_raw        = df_raw,
            df_ok         = df_ok,
            creds_path    = CREDS_PATH,
            spreadsheet_id= SPREADSHEET_ID,
            sheet_name    = sheet,
            write_back    = wb_origin_flag,
            dry_run       = not wb_origin_flag,
        )
    else:
        log.debug(
            "🔸 %s: pulando write-back de origem (já feito dentro de pipeline)",
            sheet
        )

    # 5) Preparar DataFrame de destino (modelo)
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 6) Write-back de destino  ──────────────────────────────────────────────
    if sheet.lower().startswith("ga"):
        log.info("🔸 %s: write-back de destino ignorado (Google Analytics)", sheet)
        df_dest = pd.DataFrame()      # retorna DataFrame vazio
    else:
        df_dest = write_back_for_sheet(
            df_model,
            sheet_name     = sheet,
            creds_path     = CREDS_PATH,
            spreadsheet_id = SPREADSHEET_ID,
            write_back     = wb_dest_flag,
            dry_run        = dry_run_dest,
        )
        if df_dest is None:
            df_dest = pd.DataFrame()

    # 7) Retorno
    return {"dest": df_dest, "taxo": taxo_report}


In [8]:
# %% [code]
%xmode verbose
# Cell 6: Processamento em lote das abas (com logging via logs.logging_setup)
from contextlib import suppress
import gc
import pandas as pd
from tqdm.auto import tqdm

from logs.logging_setup import get_logger
log = get_logger(__name__)

from load.dest_writer import prefetch_meta

# 1) Leitura batch de todas as abas
all_raw = fetcher.get(SHEET_NAMES)

# 2) Copia cada DataFrame para não alterar in-place
all_raw = {name: df.copy() for name, df in all_raw.items()}

# 3) Registrar colunas originais de cada aba para debug
orig_columns_map = {name: df.columns.tolist() for name, df in all_raw.items()}
for name, cols in orig_columns_map.items():
    log.debug(f"Aba '{name}' colunas originais: {cols}")

# 4) Pré-busca de cabeçalhos e IDs das abas-modelo
prefetch_meta(fetcher, SPREADSHEET_ID)
log.info("📥 Prefetch meta concluído – começando processamento das abas")

# 5) Processamento aba a aba
results: dict[str, dict[str, object]] = {}

for sheet in tqdm(SHEET_NAMES, desc="Processando abas"):
    is_ga = sheet.lower().startswith("ga")
    if is_ga:
        log.info(f"🔸 {sheet}: apenas write-back de origem; destino será ignorado")

    out = run_etl_for_sheet(
        sheet           = sheet,
        wb_origin_flag  = WRITE_BACK_ORIGIN,
        wb_dest_flag    = WRITE_BACK_DEST,
        dry_run_dest    = DRY_RUN_DEST,
        preloaded_raw   = all_raw[sheet],
    )

    results[sheet] = {"dest": out["dest"], "taxo": out["taxo"]}
    log.debug(f"Aba '{sheet}' processada – resultados armazenados")

    gc.collect()

log.info("✅ Processamento de todas as abas concluído")


16:12:26 DEBUG urllib3.connectionpool › https://sheets.googleapis.com:443 "POST /v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg/values:batchUpdate HTTP/1.1" 200 None


16:12:26 DEBUG    16:12:26 DEBUG urllib3.connectionpool › https://sheets.googleapis.com:443 "POST                       
                  /v4/spreadsheets/1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg/values:batchUpdate HTTP/1.1" 200 None

16:12:26 INFO load.origin_writer › ✅ Write-back concluído para 'metaRegiao' (10389 linhas × 17 colunas)


         INFO     16:12:26 INFO load.origin_writer › ✅ Write-back concluído para 'metaRegiao' (10389 linhas × 17       
                  colunas)

/home/debrito/Documentos/etl_debrito/load/dest_writer.py:192: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_out = df_model.reindex(columns=header, fill_value="").applymap(_scalar)
16:12:26 INFO load.dest_writer › Destino 'regiao': nenhuma linha nova para gravar


...

16:12:55 WARNING treat.utils.validations › [Validação] Coluna 'ad_name' inexistente em df_ok


         WARNING  16:12:55 WARNING treat.utils.validations › [Validação] Coluna 'ad_name' inexistente em df_ok

16:12:55 DEBUG treat.bi_param_utils › [BIParamLookup] coluna 'ad_name' ausente; skip fill_utm_content


         DEBUG    16:12:55 DEBUG treat.bi_param_utils › [BIParamLookup] coluna 'ad_name' ausente; skip fill_utm_content

AttributeError: 'str' object has no attribute 'astype'

In [ ]:
# %% [code]
# Cell 7: Validação de consistência de datas entre modelos e estatísticas de uso
from logs.logging_setup import get_logger
log = get_logger(__name__)

from treat.treat_pipeline import BIParamLookup
from treat.utils.validations import validate_consistent_dates_across_models

import gspread
import google.auth
from pprint import pprint

# ── 1) Extrair apenas os DataFrames de destino ─────────────────────────────
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# ── 2) Validar consistência de datas ───────────────────────────────────────
log.info("🔍 Validando consistência de datas entre modelos …")
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

if df_inconsistencies is not None and not df_inconsistencies.empty:
    log.warning("💥 Inconsistências encontradas:")
    display(df_inconsistencies)
else:
    log.info("✅ Nenhuma divergência de start/end entre modelos.")

# ── 3) Limpar caches se necessário ─────────────────────────────────────────
# Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher`)
fetcher.refresh(SHEET_NAMES)
# Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

# ── 4) Estatísticas de uso das planilhas ───────────────────────────────────
creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])
gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
log.info("📊 Top 10 abas que mais ocupam células:")
for cells, title, rows, cols in stats[:10]:
    log.info(f"  • {title}: {rows}×{cols} = {cells:,} células")


In [ ]:
#8
# %% [code]
from treat.treat_pipeline import BIParamLookup

# — Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher` em células anteriores)
fetcher.refresh(SHEET_NAMES)

# — Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


In [ ]:
#9
# — Forçar recarregamento dos parâmetros BI em qualquer ponto do notebook —
from treat.bi_param_utils import BIParamLookup

# Zera o cache interno para que a próxima chamada a .df() refaça o carregamento
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


In [ ]:
#10 – Validar consistência de datas entre modelos

from treat.utils.validations import validate_consistent_dates_across_models

# Extrai apenas os DataFrames de destino
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# Roda a validação
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

# Exibe resultados
if df_inconsistencies is not None and not df_inconsistencies.empty:
    display(df_inconsistencies)
else:
    print("✅ Nenhuma divergência de start/end entre modelos.")


In [ ]:
import gspread, google.auth
from pprint import pprint

creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])
gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
pprint(stats[:40])                # top 10 abas que mais ocupam células
